# DataTour — Starter Notebook ultra-minimal

Ce notebook fournit une baseline très simple : chargement des données, quelques features, entraînement sur tout le train, puis génération de `submission.csv`.

Il ne contient pas de validation temporelle, pas de comparaison de modèles, pas de graphiques et pas d’analyse avancée.


In [2]:
# DataTour — Starter Notebook ultra-minimal
# Détection de fraude Mobile Money
#
# Objectif:
# - charger les données
# - créer quelques variables simples
# - entraîner un modèle baseline sur tout le train
# - générer submission.csv

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


DATA_DIR = Path(".")  # dossier contenant train.csv, test.csv, sample_submission.csv

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

ID_COL = "id"
TARGET_COL = "fraud_flag"

print("train:", train.shape)
print("test :", test.shape)
display(train.head())


# Contrôles rapides

print("Taux de fraude:", train[TARGET_COL].mean())
print("Valeurs manquantes train:", train.isna().sum().sum())
print("Valeurs manquantes test :", test.isna().sum().sum())


# Feature engineering minimal

def add_features(df, freq_maps=None, fit=False):
    df = df.copy()

    df["amount_log1p"] = np.log1p(np.maximum(df["amount"], 0))

    df["origin_balance_change"] = df["origin_balance_after"] - df["origin_balance_before"]
    df["destination_balance_change"] = df["destination_balance_after"] - df["destination_balance_before"]

    eps = 1e-6
    df["amount_to_origin_before"] = df["amount"] / (np.abs(df["origin_balance_before"]) + eps)
    df["amount_to_destination_before"] = df["amount"] / (np.abs(df["destination_balance_before"]) + eps)

    if fit or freq_maps is None:
        freq_maps = {
            "origin_account": df["origin_account"].value_counts(normalize=True).to_dict(),
            "destination_account": df["destination_account"].value_counts(normalize=True).to_dict(),
        }

    df["origin_account_freq"] = df["origin_account"].map(freq_maps["origin_account"]).fillna(0)
    df["destination_account_freq"] = df["destination_account"].map(freq_maps["destination_account"]).fillna(0)

    
    df = df.drop(columns=["origin_account", "destination_account"])

    return df, freq_maps


# Préparation train/test

train_fe, freq_maps = add_features(train, fit=True)
test_fe, _ = add_features(test, freq_maps=freq_maps)

X_train = train_fe.drop(columns=[ID_COL, TARGET_COL])
y_train = train_fe[TARGET_COL]
X_test = test_fe.drop(columns=[ID_COL])

categorical_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", onehot, categorical_cols),
    ],
    sparse_threshold=0.0,
)

model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.06,
    max_leaf_nodes=31,
    random_state=42,
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model),
])


# Entraînement sur tout le train

pipe.fit(X_train, y_train)


# Génération de la soumission

test_pred = pipe.predict_proba(X_test)[:, 1]
test_pred = np.clip(test_pred, 0, 1)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "target": test_pred,
})

submission.to_csv("submission.csv", index=False)

display(submission.head())
print("submission.csv généré")


# Vérifications finales

assert submission.shape[0] == test.shape[0]
assert list(submission.columns) == [ID_COL, "target"]
assert submission[ID_COL].is_unique
assert submission["target"].between(0, 1).all()
assert set(submission[ID_COL]) == set(sample_submission[ID_COL])

print("Soumission valide.")


train: (1290081, 11)
test : (430100, 10)


,id,period,operation,amount,origin_account,origin_balance_before,origin_balance_after,destination_account,destination_balance_before,destination_balance_after,fraud_flag
0,dtf_0000001_ffa5beb5,0,op_05,636.75,acc_o_307358626ad66fed,87.00,-549.75,acc_d_7fac3b16af7d127b,630.88,1267.62,0
1,dtf_0000002_61992e82,0,op_05,636.12,acc_o_aeb690c57bf5d1de,76.93,76.93,acc_d_1d6120e8b117aa14,731.70,731.70,0
2,dtf_0000003_9a123b6d,0,op_05,681.00,acc_o_655c41913944d2b7,15943.74,15262.75,acc_d_ec2c21517a0ccb1a,758.83,1439.84,0
3,dtf_0000004_240f3dae,0,op_03,28175.40,acc_o_ba23a2b955a79a8b,-443.88,-28619.28,acc_d_a3dd8504815ec133,770924.84,799100.24,0
4,dtf_0000005_f18939e7,0,op_03,86429.15,acc_o_d05a23079bd066c1,-670.85,-87100.01,acc_d_0d4880267e62d5c4,91.13,86520.29,0


Taux de fraude: 0.10041384998306307
Valeurs manquantes train: 0
Valeurs manquantes test : 0


,id,target
0,dtf_0000001_08a8a524,0.000012
1,dtf_0000002_ae0d3769,0.295338
2,dtf_0000003_843bab7c,0.316001
3,dtf_0000004_91643844,0.000012
4,dtf_0000005_17bd9a08,0.303139


submission.csv généré
Soumission valide.
